In [ ]:
from pyspark.sql import DataFrame, functions as F

from transforms.fact_league_season import build_fact_league_season
from transforms.fact_match import build_fact_match

SILVER_TABLE = "stryktipset.silver.matches"
DIM_LEAGUE_TABLE = "stryktipset.gold.dim_league"
FACT_MATCH_TABLE = "stryktipset.gold.fact_match"
FACT_LEAGUE_SEASON_TABLE = "stryktipset.gold.fact_league_season"


In [ ]:
def main():
    matches = spark.table(SILVER_TABLE)
    dim_league = spark.table(DIM_LEAGUE_TABLE)

    df = build_fact_match(matches, dim_league)
    df.write.format('delta').mode('overwrite').option(
        'overwriteSchema', 'true'
    ).saveAsTable(FACT_MATCH_TABLE)

    # Aggregate fact, built from what we just wrote rather than from silver --
    # it is a summary of fact_match, so it should never be able to disagree
    # with it. Two facts sharing dim_league and dim_season: a constellation.
    league_season = build_fact_league_season(spark.table(FACT_MATCH_TABLE))
    league_season.write.format('delta').mode('overwrite').option(
        'overwriteSchema', 'true'
    ).saveAsTable(FACT_LEAGUE_SEASON_TABLE)

    # Returned as the task's run output so a scheduled run can be audited
    # from the Jobs API without opening the notebook -- same as 06 and 07.
    matches_written = spark.table(FACT_MATCH_TABLE).count()
    leagues = spark.table(FACT_MATCH_TABLE).select('league_key').distinct().count()
    seasons = league_season.count()
    print(f"fact_match: {matches_written} rows across {leagues} leagues")
    print(f"fact_league_season: {seasons} rows")

    dbutils.notebook.exit(
        f"fact_match={matches_written} leagues={leagues} league_seasons={seasons}"
    )

main()
